# [0] Phase 3 — Attention Heatmap under Semantic Perturbation

**Method**: SmoothGrad (n=10) — average gradient saliency over 10 noisy copies → per-patch importance  
**Model**: Qwen2.5-VL-3B-Instruct (bfloat16, attn_implementation='eager')  
**Images**: clean image + 18 semantic perturbations (weather / camera / illumination / occlusion)  
**Metrics**:
- Attention Drift Score = L2(clean_saliency_map, perturbed_saliency_map)
- Safety Object Attention Ratio (SOAR) = saliency on safety region / total saliency

**Output**: `experiments/results/phase3/`

In [ ]:
# [1] 환경 확인
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
DTYPE = torch.bfloat16
print(f'device: {DEVICE}, dtype: {DTYPE}')

In [ ]:
# [2] 경로 및 설정
PROJECT_ROOT = Path.cwd().resolve()
for _cand in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (_cand / 'data').is_dir() and (_cand / 'experiments').is_dir():
        PROJECT_ROOT = _cand
        break
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

CLEAN_IMAGE   = PROJECT_ROOT / 'experiments' / 'image2_infer.png'
SEMANTIC_DIR  = PROJECT_ROOT / 'experiments' / 'semantic_perturbations'
OUTPUT_DIR    = PROJECT_ROOT / 'experiments' / 'results' / 'phase3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Safety region bounding box (fraction of image: [x0, y0, x1, y1])
# image2: traffic light (upper center) + pedestrian crossing (mid area)
SAFETY_BBOX_FRACS = [
    [0.30, 0.02, 0.70, 0.50],  # traffic light / signal region
    [0.20, 0.45, 0.80, 0.80],  # pedestrian crossing region
]

# Phase 1 best prompt
BEST_PROMPT = (
    'Look at the road image and judge only whether the vehicle may proceed straight.\n\n'
    'Output:\n'
    'Decision: [Proceed / Do not proceed / Cannot determine]\n'
    'Evidence: Observation -> Impact\n\n'
    'Rules:\n'
    '- Use only what is visible in the image.\n'
    '- Avoid long explanations.\n'
    '- If uncertain, choose Cannot determine.\n\n'
)

# All perturbations: key -> filepath
PERTURBATIONS = {
    'clean':                           CLEAN_IMAGE,
    'blur':                            SEMANTIC_DIR / 'image2_blur.png',
    'noise':                           SEMANTIC_DIR / 'image2_noise.png',
    'translate_right':                 SEMANTIC_DIR / 'image2_translate_right.png',
    'occlude_top50':                   SEMANTIC_DIR / 'image2_occlude_top50.png',
    'occlude_bottom50':                SEMANTIC_DIR / 'image2_occlude_bottom50.png',
    'weather_fog_mild':                SEMANTIC_DIR / 'image2_weather_fog_mild.png',
    'weather_fog_dense':               SEMANTIC_DIR / 'image2_weather_fog_dense.png',
    'weather_rain_streaks':            SEMANTIC_DIR / 'image2_weather_rain_streaks.png',
    'weather_snow_particles':          SEMANTIC_DIR / 'image2_weather_snow_particles.png',
    'weather_dust_haze':               SEMANTIC_DIR / 'image2_weather_dust_haze.png',
    'illumination_sun_glare':          SEMANTIC_DIR / 'image2_illumination_sun_glare.png',
    'illumination_night_low_light':    SEMANTIC_DIR / 'image2_illumination_night_low_light.png',
    'camera_motion_blur':              SEMANTIC_DIR / 'image2_camera_motion_blur.png',
    'camera_defocus_blur':             SEMANTIC_DIR / 'image2_camera_defocus_blur.png',
    'camera_windshield_droplets':      SEMANTIC_DIR / 'image2_camera_windshield_droplets.png',
    'camera_jpeg_q45':                 SEMANTIC_DIR / 'image2_camera_jpeg_q45.png',
    'camera_resolution_drop_070':      SEMANTIC_DIR / 'image2_camera_resolution_drop_070.png',
    'camera_low_light_sensor_noise':   SEMANTIC_DIR / 'image2_camera_low_light_sensor_noise.png',
}

# Categories for grouped visualization
CATEGORIES = {
    'basic':        ['clean', 'blur', 'noise', 'translate_right', 'occlude_top50', 'occlude_bottom50'],
    'weather':      ['weather_fog_mild', 'weather_fog_dense', 'weather_rain_streaks',
                     'weather_snow_particles', 'weather_dust_haze'],
    'camera':       ['camera_motion_blur', 'camera_defocus_blur', 'camera_windshield_droplets',
                     'camera_jpeg_q45', 'camera_resolution_drop_070', 'camera_low_light_sensor_noise'],
    'illumination': ['illumination_sun_glare', 'illumination_night_low_light'],
}

print(f'Total images to process: {len(PERTURBATIONS)}')
for path in PERTURBATIONS.values():
    assert Path(path).exists(), f'Missing: {path}'
print('All image files found.')

In [ ]:
# [3] Qwen2.5-VL-3B 로드 (Phase 2와 동일 설정)
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
_cache_root = Path.home() / '.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots'
_snaps = sorted(_cache_root.glob('*/')) if _cache_root.exists() else []
MODEL_PATH = str(_snaps[0]) if _snaps else MODEL_ID

if 'model' not in dir() or model is None:
    print(f'Loading model from: {MODEL_PATH}')
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=DTYPE,
        attn_implementation='eager',
        local_files_only=True,
    ).to(DEVICE)
    processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    print('Model loaded and frozen.')
else:
    print('Model already loaded, reusing.')

# Full-attention block indices in the vision encoder
try:
    _vis = model.visual if hasattr(model, 'visual') else model.model.visual
    _full_att = list(_vis.config.fullatt_block_indexes)
    print(f'Vision encoder total blocks: {len(_vis.blocks)}')
    print(f'Full-attention block indices: {_full_att}')
except Exception as e:
    print(f'Vision encoder info unavailable: {e}')

In [ ]:
# [4] Gradient Saliency + Inference 함수

def _mps_cleanup():
    model.zero_grad(set_to_none=True)
    gc.collect()
    if DEVICE == 'mps':
        torch.mps.empty_cache()
    elif DEVICE == 'cuda':
        torch.cuda.empty_cache()


def _prepare_inputs(image_pil):
    """Prepare Qwen inputs from PIL image."""
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image_pil},
        {'type': 'text', 'text': BEST_PROMPT},
    ]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(msgs)
    inputs = processor(text=[text], images=image_inputs, return_tensors='pt')
    return inputs


def compute_saliency(image_pil):
    """
    Gradient saliency: d(max_logit at decision position) / d(pixel_values).
    Returns cam: (H_patches, W_patches) numpy float32 array.
    """
    inputs = _prepare_inputs(image_pil)
    input_ids      = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)
    grid_thw       = inputs['image_grid_thw'].to(DEVICE)

    # pixel_values as leaf tensor with grad enabled
    pixel_values = inputs['pixel_values'].to(DTYPE).to(DEVICE).detach().clone()
    pixel_values.requires_grad_(True)

    _mps_cleanup()

    with torch.enable_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=grid_thw,
            use_cache=False,
        )
        # Decision logit: last token position (first token to be generated)
        target_score = outputs.logits[0, -1, :].max()
        target_score.backward()

    cam = None
    if pixel_values.grad is not None:
        grad = pixel_values.grad          # (N_patches, C*temporal*patch_h*patch_w)
        importance = grad.abs().mean(dim=-1)   # (N_patches,)

        T = int(grid_thw[0, 0])
        H = int(grid_thw[0, 1])
        W = int(grid_thw[0, 2])
        cam = importance[:T * H * W].view(T, H, W).mean(dim=0)  # average temporal
        cam = cam.detach().cpu().float().numpy()
    else:
        print('  WARNING: pixel_values.grad is None — returning zero cam')
        T, H, W = int(grid_thw[0,0]), int(grid_thw[0,1]), int(grid_thw[0,2])
        cam = np.zeros((H, W), dtype=np.float32)

    _mps_cleanup()
    return cam, (int(grid_thw[0,0]), int(grid_thw[0,1]), int(grid_thw[0,2]))


def run_inference(image_pil):
    """Run greedy inference and return generated text."""
    inputs = _prepare_inputs(image_pil)
    input_ids      = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)
    grid_thw       = inputs['image_grid_thw'].to(DEVICE)
    pixel_values   = inputs['pixel_values'].to(DTYPE).to(DEVICE)

    with torch.no_grad():
        gen = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=grid_thw,
            max_new_tokens=64,
        )
    text = processor.decode(gen[0][input_ids.shape[1]:], skip_special_tokens=True)
    _mps_cleanup()
    return text



def smooth_grad_saliency(image_pil, n=10, noise_level=0.1):
    """SmoothGrad: average gradient saliency over n noisy copies (noise_level * 255 std)."""
    img_arr = np.array(image_pil).astype(np.float32)
    sigma = noise_level * 255.0
    cams = []
    grid_ref = None
    for i in range(n):
        noise = np.random.normal(0, sigma, img_arr.shape).astype(np.float32)
        noisy_pil = Image.fromarray(np.clip(img_arr + noise, 0, 255).astype(np.uint8))
        cam, grid = compute_saliency(noisy_pil)
        cams.append(cam)
        if grid_ref is None:
            grid_ref = grid
        gc.collect()
        if DEVICE == 'mps':
            torch.mps.empty_cache()
        elif DEVICE == 'cuda':
            torch.cuda.empty_cache()
        print(f'    smoothgrad {i+1}/{n}')
    print()
    return np.mean(cams, axis=0), grid_ref

print('Functions defined.')

In [ ]:
# [5] 전체 이미지 처리 (SmoothGrad n=10 + inference)
# MPS 환경에서 이미지당 ~2-4분 소요 (n=10), 총 19장 약 40-80분 예상

results = {}

for idx, (name, img_path) in enumerate(PERTURBATIONS.items()):
    t0 = time.time()
    image_pil = Image.open(img_path).convert('RGB')

    cam, grid = smooth_grad_saliency(image_pil, n=10)
    decision  = run_inference(image_pil)

    elapsed = time.time() - t0
    results[name] = {
        'cam':       cam,
        'grid':      grid,
        'decision':  decision,
        'image_pil': image_pil,
    }

    decision_short = decision.split('\n')[0][:50]
    print(f'[{idx+1:2d}/19] {name:<35} | grid {grid[1]}x{grid[2]} | {elapsed:.1f}s | {decision_short}')

print('\nAll images processed.')

In [ ]:
# [6] 지표 계산: Attention Drift Score + SOAR

def normalize_cam(cam):
    lo, hi = cam.min(), cam.max()
    return (cam - lo) / (hi - lo + 1e-8)


def cam_to_image_size(cam, target_size):
    """Upscale cam (H_p, W_p) to target_size (W_img, H_img) using bilinear."""
    cam_norm = normalize_cam(cam)
    cam_pil  = Image.fromarray((cam_norm * 255).astype(np.uint8)).resize(target_size, Image.BILINEAR)
    return np.array(cam_pil).astype(np.float32) / 255.0


def attention_drift_score(clean_cam, perturbed_cam):
    """L2 distance between normalized saliency maps (same spatial size)."""
    c = normalize_cam(clean_cam)
    p = normalize_cam(perturbed_cam)
    if c.shape != p.shape:
        p = np.array(Image.fromarray(p).resize((c.shape[1], c.shape[0]), Image.BILINEAR))
    return float(np.sqrt(((c - p) ** 2).mean()))


def soar(cam, image_pil, safety_bbox_fracs):
    """Safety Object Attention Ratio: sum saliency in safety bboxes / total."""
    H_cam, W_cam = cam.shape
    safety_mask = np.zeros((H_cam, W_cam), dtype=bool)
    for x0f, y0f, x1f, y1f in safety_bbox_fracs:
        x0 = int(x0f * W_cam); x1 = int(x1f * W_cam)
        y0 = int(y0f * H_cam); y1 = int(y1f * H_cam)
        safety_mask[y0:y1, x0:x1] = True
    total   = float(cam.sum()) + 1e-8
    safety  = float(cam[safety_mask].sum())
    return safety / total


clean_cam = results['clean']['cam']

metrics = {}
for name, data in results.items():
    drift = attention_drift_score(clean_cam, data['cam']) if name != 'clean' else 0.0
    s     = soar(data['cam'], data['image_pil'], SAFETY_BBOX_FRACS)
    decision_line = data['decision'].split('\n')[0]
    metrics[name] = {'drift': drift, 'soar': s, 'decision': decision_line}

# Print table
print(f"{'Perturbation':<35} {'Drift':>8} {'SOAR':>8}   Decision")
print('-' * 85)
for name, m in metrics.items():
    print(f"{name:<35} {m['drift']:>8.4f} {m['soar']:>8.4f}   {m['decision'][:40]}")

In [ ]:
# [7] 카테고리별 Heatmap Grid 시각화
# 각 카테고리: 2행 (원본 / saliency overlay)

CMAP = plt.cm.jet
OVERLAY_ALPHA = 0.45

def make_overlay(image_pil, cam):
    W, H = image_pil.size
    cam_up    = cam_to_image_size(cam, (W, H))
    img_arr   = np.array(image_pil).astype(np.float32) / 255.0
    heatmap   = CMAP(cam_up)[:, :, :3]
    overlay   = np.clip((1 - OVERLAY_ALPHA) * img_arr + OVERLAY_ALPHA * heatmap, 0, 1)
    return overlay, cam_up


for cat_name, cat_keys in CATEGORIES.items():
    n = len(cat_keys)
    fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7))
    if n == 1:
        axes = axes.reshape(2, 1)

    for col, name in enumerate(cat_keys):
        data     = results[name]
        m        = metrics[name]
        image_pil = data['image_pil']
        cam       = data['cam']
        overlay, cam_up = make_overlay(image_pil, cam)

        # Row 0: original image
        axes[0, col].imshow(image_pil)
        title = name if name != 'clean' else 'clean (baseline)'
        axes[0, col].set_title(title, fontsize=8, fontweight='bold' if name == 'clean' else 'normal')
        axes[0, col].axis('off')

        # Row 1: saliency overlay + metrics
        axes[1, col].imshow(overlay)
        dec = m['decision'][:30]
        if name != 'clean':
            label = f"Drift={m['drift']:.3f}\nSOAR={m['soar']:.3f}\n{dec}"
        else:
            label = f"SOAR={m['soar']:.3f}\n{dec}"
        axes[1, col].set_title(label, fontsize=7)
        axes[1, col].axis('off')

    fig.suptitle(
        f'Phase 3 — Where Does the Model Look? Saliency Heatmaps under {cat_name.upper()} Perturbations\n'
        f'(jet colormap overlay: red/yellow = high attention, blue = low attention)',
        fontsize=11
    )
    plt.tight_layout()
    out_path = OUTPUT_DIR / f'phase3_heatmap_{cat_name}_perturbations.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

In [ ]:
# [8] Clean vs Top-Drifted 비교 Summary Figure

# Sort by drift score (descending)
sorted_by_drift = sorted(
    [(k, v['drift']) for k, v in metrics.items() if k != 'clean'],
    key=lambda x: x[1], reverse=True
)
print('Top drifted perturbations:')
for name, drift in sorted_by_drift[:8]:
    print(f'  {name:<35} drift={drift:.4f}  soar={metrics[name]["soar"]:.4f}  {metrics[name]["decision"][:35]}')

# Select clean + top 5 drifted for summary
TOP_N = 5
selected_names = ['clean'] + [k for k, _ in sorted_by_drift[:TOP_N]]

fig, axes = plt.subplots(3, len(selected_names), figsize=(3.8 * len(selected_names), 11))

for col, name in enumerate(selected_names):
    data = results[name]
    m    = metrics[name]
    img  = data['image_pil']
    cam  = data['cam']
    W, H = img.size

    cam_norm = normalize_cam(cam)
    cam_up   = cam_to_image_size(cam, (W, H))
    img_arr  = np.array(img).astype(np.float32) / 255.0
    overlay  = np.clip((1 - OVERLAY_ALPHA) * img_arr + OVERLAY_ALPHA * CMAP(cam_up)[:,:,:3], 0, 1)

    # Row 0: original image
    axes[0, col].imshow(img)
    title_text = '[BASELINE]' if name == 'clean' else name
    axes[0, col].set_title(title_text, fontsize=9, fontweight='bold' if name == 'clean' else 'normal')
    axes[0, col].axis('off')

    # Row 1: standalone heatmap
    axes[1, col].imshow(cam_norm, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title('saliency map', fontsize=8)
    axes[1, col].axis('off')

    # Row 2: overlay + metrics
    axes[2, col].imshow(overlay)
    dec = m['decision'].split('\n')[0][:30]
    if name != 'clean':
        label = f"Drift={m['drift']:.3f} | SOAR={m['soar']:.3f}\n{dec}"
    else:
        label = f"SOAR={m['soar']:.3f}\n{dec}"
    axes[2, col].set_title(label, fontsize=7)
    axes[2, col].axis('off')

fig.suptitle(
    'Phase 3 — How Much Does the Model\'s Attention Shift? Clean Baseline vs Top-5 Most Drifted Perturbations\n'
    'Row 1: Input Image | Row 2: Raw Saliency Map (brighter = more important) | Row 3: Saliency Overlaid on Image',
    fontsize=11
)
plt.tight_layout()
out_path = OUTPUT_DIR / 'phase3_top5_attention_drift_comparison.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# [9] Clean baseline saliency vs Diff maps
# Show where saliency INCREASED and DECREASED after perturbation (for top-5 drifted)

top5_names = [k for k, _ in sorted_by_drift[:5]]
n = len(top5_names)

clean_cam_norm = normalize_cam(clean_cam)
clean_img  = results['clean']['image_pil']
W_c, H_c   = clean_img.size
clean_up   = cam_to_image_size(clean_cam, (W_c, H_c))

fig, axes = plt.subplots(2, n, figsize=(3.8 * n, 8))

for col, name in enumerate(top5_names):
    data = results[name]
    m    = metrics[name]
    img  = data['image_pil']
    cam  = data['cam']
    W, H = img.size

    # Upscale to same size as clean for diff computation
    pert_up = cam_to_image_size(cam, (W_c, H_c))

    diff = pert_up - clean_up  # positive = gained saliency, negative = lost saliency

    # Row 0: perturbed image overlay
    pert_overlay = np.clip(
        (1 - OVERLAY_ALPHA) * np.array(img.resize((W_c, H_c))).astype(np.float32)/255.0
        + OVERLAY_ALPHA * CMAP(pert_up)[:,:,:3], 0, 1
    )
    axes[0, col].imshow(pert_overlay)
    axes[0, col].set_title(f'{name}\nSOAR={m["soar"]:.3f}', fontsize=8)
    axes[0, col].axis('off')

    # Row 1: difference map (diverging colormap: blue=lost, red=gained)
    vmax = max(abs(diff.min()), abs(diff.max()), 0.01)
    axes[1, col].imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[1, col].set_title(f'Δsaliency (vs clean)\nDrift={m["drift"]:.3f}', fontsize=8)
    axes[1, col].axis('off')

fig.suptitle(
    'Phase 3 — Where Did Attention Increase or Decrease? Saliency Change Maps (Perturbed − Clean Baseline)\n'
    'Red = region gained attention after perturbation | Blue = region lost attention after perturbation',
    fontsize=11
)
plt.tight_layout()
out_path = OUTPUT_DIR / 'phase3_saliency_change_maps_vs_clean.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# [10] 메트릭 CSV 저장 + 최종 요약 출력
import csv

csv_path = OUTPUT_DIR / 'phase3_metrics.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['perturbation', 'category', 'drift', 'soar', 'decision'])
    writer.writeheader()
    for name, m in metrics.items():
        cat = next((c for c, keys in CATEGORIES.items() if name in keys), 'basic')
        writer.writerow({
            'perturbation': name,
            'category':     cat,
            'drift':        round(m['drift'], 4),
            'soar':         round(m['soar'], 4),
            'decision':     m['decision'],
        })

print(f'Metrics saved: {csv_path}')

# Summary by category
print('\n=== Attention Drift Score by Category ===')
for cat_name, cat_keys in CATEGORIES.items():
    cat_drifts = [metrics[k]['drift'] for k in cat_keys if k in metrics and k != 'clean']
    if cat_drifts:
        print(f'  {cat_name:<15}: mean={np.mean(cat_drifts):.4f}  max={np.max(cat_drifts):.4f}')

print('\n=== Safety Object Attention Ratio (SOAR) ===')
print(f'  clean baseline SOAR: {metrics["clean"]["soar"]:.4f}')
for cat_name, cat_keys in CATEGORIES.items():
    cat_soars = [metrics[k]['soar'] for k in cat_keys if k in metrics and k != 'clean']
    if cat_soars:
        print(f'  {cat_name:<15}: mean={np.mean(cat_soars):.4f}  min={np.min(cat_soars):.4f}')

print('\n=== Decision Changes ===')
clean_dec = metrics['clean']['decision']
print(f'  clean: {clean_dec}')
for name, m in metrics.items():
    if name != 'clean' and 'Cannot' in m['decision']:
        print(f'  {name}: {m["decision"][:50]}  [CHANGED]')
    elif name != 'clean' and 'Proceed' in m['decision'] and 'Do not' not in m['decision']:
        print(f'  {name}: {m["decision"][:50]}  [FLIPPED to Proceed!]')

print(f'\nAll Phase 3 outputs saved to: {OUTPUT_DIR}')